# Data Cleaning - HCMC Housing Dataset

Notebook này thực hiện quá trình khảo sát, làm sạch và chuẩn hóa dữ liệu nhà ở TP.HCM trước khi đưa vào các bước feature engineering và huấn luyện mô hình dự đoán giá nhà.

## Mục tiêu

- Khảo sát cấu trúc và chất lượng dữ liệu gốc.
- Kiểm tra missing values và duplicate records.
- Chuẩn hóa tên cột.
- Trích xuất District và Ward từ Address.
- Loại bỏ các feature không có dữ liệu hoặc có tỷ lệ thiếu quá cao.
- Xuất dataset đã làm sạch vào `data/processed/`.

> Các missing value còn lại chưa được impute tại bước này. Việc imputation và scaling sẽ được thực hiện trong machine learning pipeline sau khi chia train/test để tránh data leakage.

## 1. Import Libraries

In [1]:
import pandas as pd 
import numpy as np

print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)

Pandas: 3.0.6
Numpy: 2.4.6


## 2. Load Raw Dataset

In [2]:
df = pd.read_csv("../data/raw/[HCM] vietnam_housing_dataset.csv")

## 3. Data Exploration

In [3]:
print("Shape:" , df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (11779, 22)

Columns:
['Address', 'Area', 'Frontage', 'Access Road', 'House direction', 'Balcony direction', 'Floors', 'Bedrooms', 'Bathrooms', 'Legal status', 'Furniture state', 'Price', 'District', 'Ward', 'Alley_Level', 'Lat', 'Lon', 'Dist_to_Q1_km', 'School', 'Hospital / Clinic', 'Supermarket / Marketplace', 'Park']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11779 entries, 0 to 11778
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Address                    11779 non-null  str    
 1   Area                       11779 non-null  float64
 2   Frontage                   7167 non-null   float64
 3   Access Road                7261 non-null   float64
 4   House direction            3158 non-null   str    
 5   Balcony direction          1688 non-null   str    
 6   Floors                     10416 non-null  float64
 7   Bedrooms                   10181 non-null  float64
 8   Bathrooms                  9666 non-null   float64
 9   Legal status               9905 non-null   str    
 10  Furniture state            6115 non-null   str    
 11  Price                      11779 non-null  float64
 12  District                   0 non-null      float64
 13  Ward                       0 non-null      float64
 14  A

In [5]:
df.head()

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,...,District,Ward,Alley_Level,Lat,Lon,Dist_to_Q1_km,School,Hospital / Clinic,Supermarket / Marketplace,Park
0,"//, Đường Hồ Văn Long, Phường Bình Hưng Hòa B,...",48.0,4.3,4.0,NaN,NaN,1.0,2.0,2.0,Have certificate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"///, Đường Số 4, Phường Bình Hưng Hòa B, Bình ...",64.0,4.0,NaN,NaN,NaN,1.0,1.0,2.0,Have certificate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"1 /, Đường Nguyễn Văn Khối, Phường 8, Gò Vấp, ...",50.0,4.0,5.0,NaN,NaN,2.0,2.0,2.0,Have certificate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"1/ 113, Đường Lâm Thị Hố, Phường Tân Chánh Hiệ...",54.0,4.0,4.0,Đông,Đông,3.0,4.0,4.0,Have certificate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"1/ kênh 19/5, Đường 26/3, Phường Bình Hưng Hòa...",56.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,Have certificate,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Area,11779.0,66.001616,34.055052,5.0,46.0,60.00,80.00,540.0
Frontage,7167.0,4.940386,3.107954,1.0,4.0,4.15,5.00,56.0
Access Road,7261.0,7.062248,5.136641,1.0,4.0,6.00,8.00,79.0
Floors,10416.0,2.883257,1.097023,1.0,2.0,3.00,4.00,7.0
Bedrooms,10181.0,3.379923,1.307995,1.0,2.0,3.00,4.00,9.0
Bathrooms,9666.0,3.291227,1.467961,1.0,2.0,3.00,4.00,9.0
Price,11779.0,6.214147,2.096733,1.0,4.7,6.30,7.85,10.5
District,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ward,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Alley_Level,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Missing Values Analysis

Kiểm tra số lượng và tỷ lệ dữ liệu bị thiếu ở từng thuộc tính để xác định chiến lược xử lý phù hợp.

In [7]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

missing_summary

,missing_count,missing_percent
Lon,11779,100.00
Dist_to_Q1_km,11779,100.00
School,11779,100.00
Hospital / Clinic,11779,100.00
Ward,11779,100.00
District,11779,100.00
Supermarket / Marketplace,11779,100.00
Park,11779,100.00
Lat,11779,100.00
Alley_Level,11779,100.00


## 5. Duplicate Analysis

Kiểm tra các bản ghi trùng hoàn toàn trong dữ liệu gốc nhằm tránh việc cùng một mẫu dữ liệu xuất hiện nhiều lần.

In [8]:
duplicate_count = df.duplicated().sum()

print("Số dòng bị trùng hoàn toàn:", duplicate_count)
print("Tỷ lệ duplicate:", round(duplicate_count / len(df) * 100, 2), "%")

Số dòng bị trùng hoàn toàn: 1
Tỷ lệ duplicate: 0.01 %


## 6. Data Cleaning

Tạo bản sao dữ liệu để xử lý mà không thay đổi DataFrame gốc. Quá trình cleaning bao gồm loại bỏ duplicate, chuẩn hóa tên cột và chuẩn hóa dữ liệu dạng text.

In [9]:
df_clean = df.copy()

print("Raw shape:", df.shape)
print("Cleaning shape:", df_clean.shape)

Raw shape: (11779, 22)
Cleaning shape: (11779, 22)


In [10]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

after = len(df_clean)

print("Trước:", before)
print("Sau:", after)
print("Đã xóa:", before - after, "dòng duplicate")

Trước: 11779
Sau: 11778
Đã xóa: 1 dòng duplicate


In [11]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" / ", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

print(df_clean.columns.tolist())

['address', 'area', 'frontage', 'access_road', 'house_direction', 'balcony_direction', 'floors', 'bedrooms', 'bathrooms', 'legal_status', 'furniture_state', 'price', 'district', 'ward', 'alley_level', 'lat', 'lon', 'dist_to_q1_km', 'school', 'hospital_clinic', 'supermarket_marketplace', 'park']


## 7. Location Extraction

Hai thuộc tính `district` và `ward` trong dữ liệu gốc bị thiếu hoàn toàn. Tuy nhiên, thông tin này tồn tại trong `address`, vì vậy chúng được trích xuất từ chuỗi địa chỉ.

Với cấu trúc địa chỉ hiện tại, phần tử thứ hai từ cuối được xem là quận/huyện và phần tử thứ ba từ cuối được xem là phường/xã.

In [12]:
def extract_location(address):
    parts = [part.strip() for part in str(address).split(",")]

    if len(parts) >= 3:
        ward = parts[-3]
        district = parts[-2]
        return pd.Series([district, ward])

    return pd.Series([np.nan, np.nan])


df_clean[["district", "ward"]] = df_clean["address"].apply(extract_location)

In [13]:
df_clean[["address", "district", "ward"]].head(20)

,address,district,ward
0,"//, Đường Hồ Văn Long, Phường Bình Hưng Hòa B,...",Bình Tân,Phường Bình Hưng Hòa B
1,"///, Đường Số 4, Phường Bình Hưng Hòa B, Bình ...",Bình Tân,Phường Bình Hưng Hòa B
2,"1 /, Đường Nguyễn Văn Khối, Phường 8, Gò Vấp, ...",Gò Vấp,Phường 8
3,"1/ 113, Đường Lâm Thị Hố, Phường Tân Chánh Hiệ...",Quận 12,Phường Tân Chánh Hiệp
4,"1/ kênh 19/5, Đường 26/3, Phường Bình Hưng Hòa...",Bình Tân,Phường Bình Hưng Hòa
5,"1/ Phố Trần Đại Nghĩa, Xã Tân Kiên, Bình Chánh...",Bình Chánh,Xã Tân Kiên
6,"1/, Đường An Hạ, Xã Phạm Văn Hai, Bình Chánh, ...",Bình Chánh,Xã Phạm Văn Hai
7,"1/, Đường Bành Văn Trân, Phường 7, Tân Bình, H...",Tân Bình,Phường 7
8,"1/, Đường Bình Thành, Phường Bình Hưng Hòa B, ...",Bình Tân,Phường Bình Hưng Hòa B
9,"1/, Đường Bình Thành, Phường Bình Hưng Hòa B, ...",Bình Tân,Phường Bình Hưng Hòa B


In [14]:
print("District missing:", df_clean["district"].isna().sum())
print("Ward missing:", df_clean["ward"].isna().sum())

print(df_clean["district"].value_counts(dropna=False))

print(df_clean["ward"].value_counts(dropna=False).head(30))

District missing: 0
Ward missing: 0
district
Gò Vấp               1373
Bình Tân             1162
Thủ Đức              1145
Tân Phú              1016
Quận 12               940
Tân Bình              851
Bình Thạnh            851
Quận 9                713
Quận 7                642
Quận 8                447
Phú Nhuận             376
Nhà Bè                359
Quận 10               293
Quận 6                246
Hóc Môn               244
Quận 3                222
Bình Chánh            209
Quận 2                196
Quận 4                139
Quận 11               136
Quận 1                104
Quận 5                 60
Củ Chi                 43
Thành phố Thủ Đức       9
Cần Giờ                 2
Name: count, dtype: int64
ward
Phường 5                  435
Phường 11                 433
Phường 12                 413
Phường 14                 385
Phường 13                 340
Phường 15                 336
Phường 10                 310
Phường 3                  309
Phường 4                  238
Phườ

In [15]:
text_cols = [
    "house_direction",
    "balcony_direction",
    "legal_status",
    "furniture_state",
    "district",
    "ward"
]

for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip()

### Validate extracted locations

Kiểm tra danh sách quận/huyện sau khi trích xuất nhằm phát hiện các giá trị bất thường do lỗi phân tích địa chỉ.

In [16]:
sorted(df_clean["district"].dropna().unique())

['Bình Chánh',
 'Bình Thạnh',
 'Bình Tân',
 'Cần Giờ',
 'Củ Chi',
 'Gò Vấp',
 'Hóc Môn',
 'Nhà Bè',
 'Phú Nhuận',
 'Quận 1',
 'Quận 10',
 'Quận 11',
 'Quận 12',
 'Quận 2',
 'Quận 3',
 'Quận 4',
 'Quận 5',
 'Quận 6',
 'Quận 7',
 'Quận 8',
 'Quận 9',
 'Thành phố Thủ Đức',
 'Thủ Đức',
 'Tân Bình',
 'Tân Phú']

## 8. Feature Selection

- `balcony_direction` bị loại do tỷ lệ missing khoảng 85.67%, quá cao để sử dụng đáng tin cậy.
- `alley_level`, `lat`, `lon`, `dist_to_q1_km`, `school`, `hospital_clinic`, `supermarket_marketplace` và `park` bị loại vì 100% giá trị bị thiếu.
- `district` và `ward` được giữ lại sau khi trích xuất từ `address`.
- Các missing value còn lại chưa được impute ở bước này để tránh data leakage trong quá trình train/test.

In [17]:
df_clean = df_clean.drop(columns=["balcony_direction"])

print("Shape:", df_clean.shape)

Shape: (11778, 21)


In [18]:
placeholder_cols = [
    "alley_level",
    "lat",
    "lon",
    "dist_to_q1_km",
    "school",
    "hospital_clinic",
    "supermarket_marketplace",
    "park"
]

df_clean = df_clean.drop(columns=placeholder_cols)

print("Shape:", df_clean.shape)

Shape: (11778, 13)


## 9. Final Validation

Kiểm tra cấu trúc, missing values, categorical values và duplicate sau khi hoàn tất quá trình cleaning.

In [19]:
print("Final shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nMissing values:")
print(df_clean.isnull().sum().sort_values(ascending=False))

Final shape: (11778, 13)

Columns:
['address', 'area', 'frontage', 'access_road', 'house_direction', 'floors', 'bedrooms', 'bathrooms', 'legal_status', 'furniture_state', 'price', 'district', 'ward']

Missing values:
house_direction    8620
furniture_state    5664
frontage           4611
access_road        4518
bathrooms          2113
legal_status       1874
bedrooms           1598
floors             1363
address               0
area                  0
price                 0
district              0
ward                  0
dtype: int64


In [20]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 1


In [21]:
for col in [
    "house_direction",
    "legal_status",
    "furniture_state",
    "district"
]:
    print(f"\n===== {col} =====")
    print(df_clean[col].value_counts(dropna=False))


===== house_direction =====
house_direction
<NA>          8620
Đông - Nam     601
Đông - Bắc     424
Tây - Nam      417
Tây - Bắc      369
Đông           365
Tây            358
Bắc            319
Nam            305
Name: count, dtype: Int64

===== legal_status =====
legal_status
Have certificate    9754
<NA>                1874
Sale contract        150
Name: count, dtype: Int64

===== furniture_state =====
furniture_state
<NA>     5664
Full     4090
Basic    2024
Name: count, dtype: Int64

===== district =====
district
Gò Vấp               1373
Bình Tân             1162
Thủ Đức              1145
Tân Phú              1016
Quận 12               940
Tân Bình              851
Bình Thạnh            851
Quận 9                713
Quận 7                642
Quận 8                447
Phú Nhuận             376
Nhà Bè                359
Quận 10               293
Quận 6                246
Hóc Môn               244
Quận 3                222
Bình Chánh            209
Quận 2                196
Quận 4

In [22]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

after = len(df_clean)

print("Trước khi xóa duplicate cuối:", before)
print("Sau khi xóa:", after)
print("Đã xóa thêm:", before - after)
print("Duplicate rows còn lại:", df_clean.duplicated().sum())

Trước khi xóa duplicate cuối: 11778
Sau khi xóa: 11777
Đã xóa thêm: 1
Duplicate rows còn lại: 0


## 10. Export Processed Dataset

Dataset sau khi làm sạch được lưu vào `data/processed/` để sử dụng cho các bước feature engineering và huấn luyện mô hình.

In [23]:
output_path = "../data/processed/hcm_housing_clean.csv"

df_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)

Saved: ../data/processed/hcm_housing_clean.csv


In [24]:
df_check = pd.read_csv("../data/processed/hcm_housing_clean.csv")

print("Shape:", df_check.shape)
print("Duplicate rows:", df_check.duplicated().sum())

df_check.head()

Shape: (11777, 13)
Duplicate rows: 0


,address,area,frontage,access_road,house_direction,floors,bedrooms,bathrooms,legal_status,furniture_state,price,district,ward
0,"//, Đường Hồ Văn Long, Phường Bình Hưng Hòa B,...",48.0,4.3,4.0,NaN,1.0,2.0,2.0,Have certificate,Basic,3.00,Bình Tân,Phường Bình Hưng Hòa B
1,"///, Đường Số 4, Phường Bình Hưng Hòa B, Bình ...",64.0,4.0,NaN,NaN,1.0,1.0,2.0,Have certificate,Basic,4.40,Bình Tân,Phường Bình Hưng Hòa B
2,"1 /, Đường Nguyễn Văn Khối, Phường 8, Gò Vấp, ...",50.0,4.0,5.0,NaN,2.0,2.0,2.0,Have certificate,Full,5.30,Gò Vấp,Phường 8
3,"1/ 113, Đường Lâm Thị Hố, Phường Tân Chánh Hiệ...",54.0,4.0,4.0,Đông,3.0,4.0,4.0,Have certificate,Full,4.55,Quận 12,Phường Tân Chánh Hiệp
4,"1/ kênh 19/5, Đường 26/3, Phường Bình Hưng Hòa...",56.0,NaN,NaN,NaN,5.0,NaN,NaN,Have certificate,NaN,6.90,Bình Tân,Phường Bình Hưng Hòa


## Cleaning Summary

Dataset sau quá trình làm sạch:

- Dữ liệu gốc: 11,779 dòng, 22 cột.
- Dữ liệu sau xử lý: 11,777 dòng, 13 cột.
- Duplicate rows cuối cùng: 0.
- Target `price` không có missing value.
- `district` và `ward` được trích xuất từ `address`.
- `balcony_direction` bị loại do tỷ lệ missing quá cao.
- Các feature rỗng hoàn toàn được loại bỏ.
- Các missing value còn lại được giữ lại để xử lý trong ML preprocessing pipeline sau khi chia train/test.

Processed dataset:

`data/processed/hcm_housing_clean.csv`